In [0]:
import warnings
warnings.filterwarnings('ignore')

In [0]:
from crewai import Agent, Task, Crew

In [0]:
#%pip install -U langchain-google-genai

In [0]:
#%pip install langchain-google-genai==0.0.9

In [0]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    streaming=False
)


In [0]:
Issue_Classification_Agent=Agent(
    role="Issue Classification Agent",
    goal="Identify and classify customer issues accurately",
    backstory="You analyze customer messages and extract structured issue details.",
    llm=llm,
    verbose=True
    
)

Policy_Reasoning_Agent=Agent(
    role="Policy Interpretation agent and knowledge base",
    goal="Match customer issues with business policies",
    backstory="You reason over internal policies and determine allowed actions.",
    llm=llm,
    verbose=True
    
)

Resolution_Agent=Agent(
    role="resolution recommendation agent",
    goal="generate a clear and empathetic customer response",
    backstory="You craft user-friendly responses based on policies and issue context.",
    llm=llm,
    verbose=True,
    max_iter=3
)

Escalation_Agent=Agent(
    role="Escalation reasoning",
    goal="make the final decision on escalation",
    backstory="You are the final authority deciding if human intervention is required.",
    llm=llm,
    verbose=True
    
)

In [0]:
identify_issue = Task(
    description="""Analyze the customer message: "{customer_query}".

Return STRICT JSON:
{{
  "issue_type": "delivery_damage | refund | return | exchange | other",
  "urgency": "low | normal | high",
  "sentiment": "angry | neutral | confused",
  "summary": "one sentence summary",
  "confidence": 0.0 to 1.0
}}
""",
    agent=Issue_Classification_Agent,
    expected_output="Structured issue classification JSON."
)

fetch_policy = Task(
    description="""Based on the identified issue type, determine the applicable policy.

Rules:
- If issue_type is "delivery_damage":
  - Policy: Damaged items during delivery are eligible for replacement or refund within 7 days
  - Allowed actions: refund, replacement

- If issue_type is "refund":
  - Policy: Refunds are processed within 5–7 business days
  - Allowed actions: refund

- If issue_type is "return":
  - Policy: Returns allowed within 10 days if unused and undamaged
  - Allowed actions: return

- Otherwise:
  - NO MATCH FOUND

Return STRICT JSON:
{{
  "matched_policy": "policy text or NO MATCH FOUND",
  "allowed_actions": ["refund", "replacement", "return"]
}}
""",
    agent=Policy_Reasoning_Agent,
    expected_output="JSON with matched policy and allowed actions."
)

generate_resolution = Task(
    description="""Using:
- Issue classification
- Matched policy
- Allowed actions

Create a customer-ready response.

Return STRICT JSON:
{{
  "customer_message": "polite and empathetic reply",
  "action_taken": "refund | replacement | return | none",
  "next_steps": "clear next steps for customer"
}}

- You MUST NOT decide or mention escalation status.
- Do NOT use terms like ‘escalate’, ‘requires escalation’, or ‘human review’.
- Only provide resolution details and confidence.”
""",
    agent=Resolution_Agent,
    expected_output="Structured customer response."
)

apply_escalation = Task(
    description="""You are the FINAL decision maker.

Escalate ONLY if:
- Policy is 'NO MATCH FOUND'
- Confidence < 0.6
- Urgency is high and resolution is uncertain
- Required details (order ID, proof) are missing
- previous agent says escalation
- human safety is compromised

Return STRICT JSON:
{{
  "final_status": "resolved | requires_escalation",
  "reason": "brief justification"
}}
""",
    agent=Escalation_Agent,
    expected_output="Final escalation decision."
)

In [0]:

Ecommerce_Crew = Crew(
    agents=[
        Issue_Classification_Agent,
        Policy_Reasoning_Agent,
        Resolution_Agent,
        Escalation_Agent
    ],
    tasks=[
        identify_issue,
        fetch_policy,
        generate_resolution,
        apply_escalation
    ],
    verbose=True
)

In [0]:
customer_query = """
Ordered chicken pickle from X which is in India to be delevered in the US-Florida. Ordered the package on 14th january and its 9th February still not recieved the package. can you please let me know what happened and send the package
"""

result = Ecommerce_Crew.kickoff(
    inputs={"customer_query": customer_query}
)

print("\n--- FINAL SYSTEM OUTPUT ---\n")
print(result)

#

In [0]:
#I ordered the Aqualogica Vitamin C serum from Amazon, but the bottle came broken.I want a replacement or refund immediately. This is clearly a delivery issue.

#i have received the an empty package when i ordered anarkali set from Biba. Dont know know what exactly happened. it is not expected from such a reputed brand. Can you help me know what happened with my package and send the product ASAP.

#i have ordered BP tablets from Apollo Pharmacy online and received the expired one and my mother is unwell now and admitted in hospital. How can people be such irresponsible on things that cost a life. Who will be responsible if anything happened to my mother. I want to take an action on this can somebody help?

#Driver misbehavior when used the UBER cab service to commute from office to home. Luckily escaped from the situation as the police arrived. Dont know how you compromise girls safety by hiring some stupid people. i need an answer from your manager, what if something happened to me?

#ordered 'paneer butter masala' from your restraunt and recieved 'Chicken butter masala'. Receiving a non veg food being a veg family is disgusting that too on vaikunta ekadashi just because of your mistake. need full refund ASAP and not ordering food from your restraunt again

#Ordered chicken pickle from X which is in India to be delevered in the US-Florida. Ordered the package on 14th january and its 9th February still not recieved the package. can you please let me know what happened and send the package